# Martirossyan Complex-Crystal Dataset Inspection

This notebook has two parts:

1. Visual inspection of the transferred MDF/ACDC GSD trajectories.
2. A first q_l/q_ls-style baseline: compute Steinhardt q_l values and test how well they separate the different crystal-structure folders.

The analysis is intentionally lightweight. Increase the sampling constants if you want a more expensive run.

In [ ]:

from pathlib import Path
import json
import math
import random
import sys

import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": True})


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "lammps").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find autoencode_statmech repo root from current working directory")


REPO_ROOT = find_repo_root()
OTHER_DATA = REPO_ROOT / "lammps/LJ/statistically_independent_samples/statistically_independent_samples/other_data"
print(f"Repo root: {REPO_ROOT}")
print(f"Other data: {OTHER_DATA}")

try:
    import gsd.hoomd
except ImportError as exc:
    raise ImportError("This notebook needs gsd: python -m pip install gsd") from exc

try:
    import freud
except ImportError as exc:
    raise ImportError("This notebook needs freud: conda install -c conda-forge freud or python -m pip install freud-analysis") from exc

DATA_DIR = OTHER_DATA / "martirossyan_complex_crystals"
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Missing Martirossyan data directory: {DATA_DIR}")

STRUCTURE_DIRS = sorted(p for p in DATA_DIR.iterdir() if p.is_dir() and (p / "dump.gsd").exists())
print(f"Found {len(STRUCTURE_DIRS)} trajectory folders")
for path in STRUCTURE_DIRS:
    print("-", path.name)


## Dataset Overview

In [ ]:

def open_gsd(path):
    try:
        return gsd.hoomd.open(name=str(path), mode="r")
    except TypeError:
        return gsd.hoomd.open(str(path), "r")


def summarize_gsd(path):
    with open_gsd(path) as traj:
        first = traj[0]
        last = traj[-1]
        box = np.asarray(first.configuration.box, dtype=float)
        return {
            "structure": path.parent.name,
            "frames": len(traj),
            "particles_first": int(first.particles.N),
            "particles_last": int(last.particles.N),
            "box_first": tuple(np.round(box[:3], 3)),
            "size_mb": path.stat().st_size / 1024**2,
        }

summary = [summarize_gsd(path / "dump.gsd") for path in STRUCTURE_DIRS]
print(f"{'structure':24s} {'frames':>8s} {'N0':>7s} {'Nlast':>7s} {'size MB':>9s} {'box Lx,Ly,Lz':>20s}")
for row in summary:
    print(f"{row['structure']:24s} {row['frames']:8d} {row['particles_first']:7d} {row['particles_last']:7d} {row['size_mb']:9.2f} {str(row['box_first']):>20s}")


## Visualize Trajectory Frames

The plots below show sampled particles from selected structure folders. Use `FRAME_INDEX = 0` for early/cooling-start frames and `FRAME_INDEX = -1` for late frames.

In [ ]:

RNG = np.random.default_rng(7)
FRAME_INDEX = -1
MAX_POINTS = 2500
STRUCTURES_TO_PLOT = [p.name for p in STRUCTURE_DIRS[:6]]


def read_frame(structure, frame_index=-1):
    path = DATA_DIR / structure / "dump.gsd"
    with open_gsd(path) as traj:
        snap = traj[frame_index]
        positions = np.asarray(snap.particles.position, dtype=float)
        box = np.asarray(snap.configuration.box, dtype=float)
    return positions, box


def set_axes_equal(ax, positions):
    mins = positions.min(axis=0)
    maxs = positions.max(axis=0)
    centers = 0.5 * (mins + maxs)
    radius = 0.55 * float(np.max(maxs - mins))
    for dim, setter in enumerate((ax.set_xlim, ax.set_ylim, ax.set_zlim)):
        setter(centers[dim] - radius, centers[dim] + radius)


def plot_structures(structure_names, frame_index=-1, max_points=2500):
    n = len(structure_names)
    cols = min(3, n)
    rows = math.ceil(n / cols)
    fig = plt.figure(figsize=(4.8 * cols, 4.2 * rows))
    for i, structure in enumerate(structure_names, start=1):
        positions, box = read_frame(structure, frame_index=frame_index)
        if len(positions) > max_points:
            idx = RNG.choice(len(positions), size=max_points, replace=False)
            show = positions[idx]
        else:
            show = positions
        ax = fig.add_subplot(rows, cols, i, projection="3d")
        sc = ax.scatter(show[:, 0], show[:, 1], show[:, 2], c=show[:, 2], s=4, cmap="viridis", alpha=0.75)
        ax.set_title(f"{structure}\nframe {frame_index}, N={len(positions)}")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("z")
        set_axes_equal(ax, show)
    fig.tight_layout()
    return fig

plot_structures(STRUCTURES_TO_PLOT, frame_index=FRAME_INDEX, max_points=MAX_POINTS)


## Compute q_l Features

This samples particles from a small number of frames per structure and computes per-particle Steinhardt q_l values. The class label is the structure folder name. This is a direct check of whether q_l descriptors separate the crystal classes.

In [ ]:

L_LIST = [4, 6, 8, 10, 12]
NUM_NEIGHBORS = 12
AVERAGE_Q = False
MAX_FRAMES_PER_STRUCTURE = 8
MAX_PARTICLES_PER_FRAME = 350
RANDOM_SEED = 11

rng = np.random.default_rng(RANDOM_SEED)


def freud_box_from_hoomd(box_array):
    box_array = np.asarray(box_array, dtype=float)
    Lx, Ly, Lz = box_array[:3]
    xy, xz, yz = (box_array[3:6] if len(box_array) >= 6 else (0.0, 0.0, 0.0))
    return freud.box.Box(Lx=Lx, Ly=Ly, Lz=Lz, xy=xy, xz=xz, yz=yz)


def sampled_frame_indices(n_frames, max_frames):
    if n_frames <= max_frames:
        return list(range(n_frames))
    return sorted(set(np.linspace(0, n_frames - 1, max_frames, dtype=int).tolist()))


def compute_q_for_frame(snapshot, order):
    positions = np.asarray(snapshot.particles.position, dtype=np.float32)
    box = freud_box_from_hoomd(snapshot.configuration.box)
    order.compute(system=(box, positions), neighbors={"num_neighbors": NUM_NEIGHBORS})
    return np.asarray(order.particle_order, dtype=np.float32)


def build_q_dataset():
    order = freud.order.Steinhardt(l=L_LIST, average=AVERAGE_Q, wl=False)
    X, y, frame_ids, particle_ids = [], [], [], []
    for structure_dir in STRUCTURE_DIRS:
        structure = structure_dir.name
        with open_gsd(structure_dir / "dump.gsd") as traj:
            for frame_index in sampled_frame_indices(len(traj), MAX_FRAMES_PER_STRUCTURE):
                q = compute_q_for_frame(traj[frame_index], order)
                if len(q) > MAX_PARTICLES_PER_FRAME:
                    idx = rng.choice(len(q), size=MAX_PARTICLES_PER_FRAME, replace=False)
                else:
                    idx = np.arange(len(q))
                X.append(q[idx])
                y.extend([structure] * len(idx))
                frame_ids.extend([frame_index] * len(idx))
                particle_ids.extend(idx.tolist())
    X = np.vstack(X)
    y = np.asarray(y)
    meta = {"frame": np.asarray(frame_ids), "particle": np.asarray(particle_ids)}
    return X, y, meta

X_q, y_q, meta_q = build_q_dataset()
print("X_q shape:", X_q.shape)
print("labels:", {label: int(np.sum(y_q == label)) for label in sorted(set(y_q))})


## q_l Separation Plots

In [ ]:

# Pair plot for the most common baseline view.
q4_idx = L_LIST.index(4)
q6_idx = L_LIST.index(6)
fig, ax = plt.subplots(figsize=(7, 5))
for label in sorted(set(y_q)):
    mask = y_q == label
    take = np.where(mask)[0]
    if len(take) > 700:
        take = rng.choice(take, size=700, replace=False)
    ax.scatter(X_q[take, q4_idx], X_q[take, q6_idx], s=8, alpha=0.55, label=label)
ax.set_xlabel("q4")
ax.set_ylabel("q6")
ax.set_title("Per-particle q4/q6 by Martirossyan structure")
ax.legend(markerscale=2, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()

# PCA over the full q_l vector.
X_scaled = StandardScaler().fit_transform(X_q)
X_pca = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X_scaled)
fig, ax = plt.subplots(figsize=(7, 5))
for label in sorted(set(y_q)):
    mask = y_q == label
    take = np.where(mask)[0]
    if len(take) > 700:
        take = rng.choice(take, size=700, replace=False)
    ax.scatter(X_pca[take, 0], X_pca[take, 1], s=8, alpha=0.55, label=label)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA of sampled q_l vectors")
ax.legend(markerscale=2, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()


## q_l Classifier Baseline

A random forest is intentionally simple here: the point is not to build the best classifier, but to quantify whether these q_l features carry enough information to separate the structure labels.

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_q,
    y_q,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_q,
)

clf = RandomForestClassifier(
    n_estimators=250,
    min_samples_leaf=3,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))
labels = sorted(set(y_q))
cm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, xticks_rotation=90, values_format=".2f", cmap="Blues")
ax.set_title("q_l random-forest baseline, normalized confusion matrix")
fig.tight_layout()

importances = clf.feature_importances_
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar([f"q{l}" for l in L_LIST], importances)
ax.set_ylabel("feature importance")
ax.set_title("Random-forest q_l feature importance")
fig.tight_layout()
